# **EDA previo a la limpieza**

El objetivo es tener un primer acercamiento a los datasets con los que se estará trabajando. Identificar variables o puntos que puedan ser problemáticos en el proceso de limpieza.

**Setup y carga del crudo**

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

DIR_RAW = Path.cwd().parent / "data" / "raw"
if not DIR_RAW.exists():
    DIR_RAW = Path.cwd() / "data" / "raw"

archivos = sorted(DIR_RAW.glob("diversificado_*.csv"))
partes = []
for f in archivos:
    parte = pd.read_csv(f, dtype=str)
    parte.insert(0, "ARCHIVO", f.stem)  # solo en memoria, no se escribe a data/raw/
    partes.append(parte)
df = pd.concat(partes, ignore_index=True)
print(f"{len(archivos)} archivos cargados -> {df.shape[0]} filas, {df.shape[1]} columnas")

**a. Numero de registros y variables**

In [ ]:
cols_negocio = [c for c in df.columns if c != "ARCHIVO"]
print(f"Registros (filas): {df.shape[0]}")
print(f"Variables (columnas): {len(cols_negocio)}")

por_departamento = (
    df.groupby("DEPARTAMENTO", sort=False)
    .size()
    .rename("registros")
    .reset_index()
    .sort_values("registros", ascending=False)
)
por_departamento

**b. Tipo de dato de cada variable**

In [ ]:
def tipo_logico(serie):
    """Heuristica simple: numerico > fecha > pocas categorias > texto libre."""
    valores = serie.dropna()
    if valores.empty:
        return "vacio"
    if pd.to_numeric(valores, errors="coerce").notna().all():
        return "numerico"
    if pd.to_datetime(valores, errors="coerce", format="mixed").notna().all():
        return "fecha"
    if valores.nunique() <= 20:
        return "categorico"
    return "texto libre"

resumen_tipos = pd.DataFrame({
    "dtype_pandas": df[cols_negocio].dtypes.astype(str),
    "tipo_logico_inferido": [tipo_logico(df[c]) for c in cols_negocio],
})
resumen_tipos

**c. Cantidad y porcentaje de valores faltantes por variable**

In [ ]:
faltantes = df[cols_negocio].isna().sum().rename("faltantes")
faltantes_pct = (faltantes / len(df) * 100).round(2).rename("faltantes_%")
resumen_faltantes = (
    pd.concat([faltantes, faltantes_pct], axis=1)
    .sort_values("faltantes", ascending=False)
)
resumen_faltantes

**d. Cantidad de valores unicos**

In [ ]:
unicos = df[cols_negocio].nunique().rename("valores_unicos").sort_values(ascending=False)
unicos_df = unicos.reset_index().rename(columns={"index": "variable"})
print(f"CODIGO es llave unica: {df['CODIGO'].nunique() == len(df)} "
      f"({df['CODIGO'].nunique()} unicos / {len(df)} filas)")
unicos_df

**e. Cantidad de registros duplicados exactos**

In [ ]:
dup_filas = df[cols_negocio].duplicated().sum()
dup_codigo = df["CODIGO"].duplicated().sum()
print(f"Duplicados exactos (fila completa): {dup_filas}")
print(f"Duplicados por CODIGO (llave de negocio): {dup_codigo}")

if dup_codigo > 0:
    df[df["CODIGO"].duplicated(keep=False)].sort_values("CODIGO")

**f. Variables con valores fuera de dominio o inconsistentes**

In [ ]:
cols_categoricas = ["NIVEL", "SECTOR", "AREA", "STATUS", "MODALIDAD", "JORNADA", "PLAN"]

for c in cols_categoricas:
    print(f"--- {c} ({df[c].nunique()} categorias) ---")
    display(df[c].value_counts(dropna=False).rename("conteo").reset_index().rename(columns={"index": c}))

**g. Variables con formatos inconsistentes**

In [ ]:
import re

PATRON_CODIGO = re.compile(r"^\d{2}-\d{2}-\d{4}-\d{2}$")
PATRON_DISTRITO_LARGO = re.compile(r"^\d{2}-\d{2}-\d{4}$")
PATRON_DISTRITO_CORTO = re.compile(r"^\d{2}-\d{3}$")
PATRON_TELEFONO_SIMPLE = re.compile(r"^\d{8}$")

print("CODIGO no calza con ##-##-####-##:", (~df["CODIGO"].str.match(PATRON_CODIGO)).sum())

distrito = df["DISTRITO"].dropna()
formato_distrito = pd.Series(
    [
        "largo (##-##-####)" if PATRON_DISTRITO_LARGO.match(v)
        else "corto (##-###)" if PATRON_DISTRITO_CORTO.match(v)
        else "otro"
        for v in distrito
    ]
).value_counts()
print("\nFormatos de DISTRITO (no nulos):")
display(formato_distrito)

telefono = df["TELEFONO"].dropna()
no_simple = ~telefono.str.match(PATRON_TELEFONO_SIMPLE)
print(f"\nTELEFONO no numerico de 8 digitos: {no_simple.sum()} de {len(telefono)} no nulos")
telefono[no_simple].head(10)

**h. Identificacion de problemas potenciales de calidad de datos**

- **Faltantes en datos de contacto/direccion**: DIRECTOR, TELEFONO, SUPERVISOR y DISTRITO son las variables con mas nulos (seccion c). Ej: DIRECTOR falta en 1732 de 11867 filas.
- **AREA** incluye la categoria "SIN ESPECIFICAR", un no-dato disfrazado de categoria valida (seccion f). Ej: un establecimiento con `AREA = "SIN ESPECIFICAR"` en vez de nulo o URBANA/RURAL.
- **PLAN** tiene categorias solapadas (SEMIPRESENCIAL y sus 3 variantes con parentesis) que probablemente deban unificarse (seccion f). Ej: `"SEMIPRESENCIAL"` vs `"SEMIPRESENCIAL (FIN DE SEMANA)"` describen esencialmente lo mismo.
- **TELEFONO** no tiene formato uniforme: mezcla numeros simples de 8 digitos con pares separados por guion o espacio (seccion g). Ej: `"22202870-73"` y `"2517748 2384159"` junto a valores normales como `"22512759"`.
- **DISTRITO** mezcla dos formatos de codigo (##-##-#### y ##-###), ademas de tener nulos (seccion g). Ej: `"01-01-0026"` vs `"01-102"` para el mismo tipo de dato.